In [1]:
def interpolate(vals):
    pts = list(zip(H, [Fr(v) for v in vals]))
    return R.lagrange_polynomial(pts)

def smul(scalar, point):
    scalar = ZZ(Fp(scalar))
    result = point - point
    t = point
    while scalar > 0:
        if scalar % 2:
            result += t
        t += t
        scalar //= 2
    return result
    
def commit(poly, srs):
    assert poly.degree() <= len(srs) - 1
    coeffs = poly.coefficients(sparse=False)
    return sum((smul(c, srs[i]) for i,c in enumerate(coeffs)), E(0))

def split_t(t, n):
    assert t == 0 or t.degree() <= 3*n + 5           # fits 3 parts of degree n+1
    return [sum(t[i]*x^(i - j*(n+2))
                for i in range(j*(n+2), min((j+1)*(n+2), t.degree()+1)))
            for j in range(3)]

In [2]:
p = 101 # Prime order
r = 17 # Group order
k = 2 # Embedding degree

Fp = GF(p)
Fr = GF(r)

# E(a,b) : y^2 = x^3 + ax + b
E = EllipticCurve(Fp, [0,3]) # y^2 = x^3 + 3 (mod 101)
# 102 : 1, 2, 3, 6, 17, 34, 51, 102

R.<x> = PolynomialRing(Fr)
K.<t> = GF(p^k, modulus=x^2 - 2)
EK = E.base_extend(K)

G1 = E(91, 35) # order = 17
G2 = EK(74,10*t) # order = 17

In [3]:
n = 4
k1 = Fr(2)
k2 = Fr(3)

w4 = Fr.multiplicative_generator()^((r-1)//n)

H = [w4**i for i in range(n)]
k1H = [k1*(w4**i) for i in range(n)]
k2H = [k2*(w4**i) for i in range(n)]

ZH = x^n - 1

In [4]:
a_vals = [4,1,13,1]      # 1*H
b_vals = [8,2,9,2]       # k1*H
c_vals = [14,3,14,3]     # k2*H

qL = [0,0,0,1]
qR = [0,0,0,1]
qM = [10,10,10,0]
qC = [0,0,0,0]
qO = [-1,-1,-1,-1]

s1 = [1,4,16,13]        # permutes 1*H
s2 = [2,8,15,9]         # permutes k1*H
s3 = [14,5,3,12]        # permutes k2*H

In [5]:
d = n + 2
s = Fr.multiplicative_generator()
srs1 = [smul(s^i, G1) for i in range(d + 1)]
srs2_1 = G2
srs2_s = smul(s, G2)

# print(srs1, srs2_1, srs2_s)

In [6]:
pl = [a_vals, b_vals, c_vals, qL, qR, qM, qC, qO, s1, s2, s3]
a_poly, b_poly, c_poly, qL_p, qR_p, qM_p, qC_p, qO_p, s1_p, s2_p, s3_p = map(interpolate, pl)

In [7]:
beta = Fr(5)
gamma = Fr(7)

alpha = Fr(4)
zeta = Fr(5)

v = Fr(9)

u = Fr(8)

In [8]:
# Round 1
b1, b2, b3, b4, b5, b6 = [ Fr.random_element() for _ in range(6)]

a_poly = (b1 * x + b2) * ZH + interpolate(a_vals)
b_poly = (b3 * x + b4) * ZH + interpolate(b_vals)
c_poly = (b5 * x + b6) * ZH + interpolate(c_vals)

print("a(x) =" ,a_poly)
print("b(x) =" ,b_poly)
print("c(x) =" ,c_poly)

C_a = commit(a_poly, srs1)
C_b = commit(b_poly, srs1)
C_c = commit(c_poly, srs1)

print(C_a, C_b, C_c)

a(x) = 4*x^5 + 3*x^4 + 2*x^3 + 8*x^2 + 15*x + 6
b(x) = 9*x^5 + 12*x^4 + 4*x^3 + 16*x^2 + 12*x + 6
c(x) = 13*x^5 + 15*x^4 + 14*x^2 + 4*x + 2
(26 : 45 : 1) (26 : 56 : 1) (12 : 69 : 1)


In [9]:
# Round 2

b7, b8, b9 = [ Fr.random_element() for _ in range(3)]

def num(i):
    return(
        (a_vals[i] + beta * H[i] + gamma) *
        (b_vals[i] + beta * k1 * H[i] + gamma) *
        (c_vals[i] + beta * k2 * H[i] + gamma)
    )

def den(i):
    return(
        (a_vals[i] + beta * s1[i] + gamma) *
        (b_vals[i] + beta * s2[i] + gamma) *
        (c_vals[i] + beta * s3[i] + gamma) 
    )
    
Z_vals = [Fr(1)]
for i in range(n):
    Z_vals.append(Z_vals[-1] * num(i) / den(i))

z_poly = (b7 * x^2 + b8 * x + b9) * ZH + interpolate(Z_vals[:n])
print("z(x) = ", z_poly)
C_z = commit(z_poly, srs1)
print(C_z)

z(x) =  12*x^6 + 7*x^5 + 2*x^4 + 11*x + 3
(1 : 2 : 1)


In [10]:
# Round 3

b10, b11, b12 = [ Fr.random_element() for _ in range(3)]

L1 = interpolate([1, 0, 0, 0])

PI_p = interpolate([0,0,0,0])

gate_part = (
    a_poly * b_poly * qM_p +
    a_poly * qL_p +
    b_poly * qR_p +
    c_poly * qO_p + 
    PI_p + 
    qC_p )

perm_n = (
    (a_poly + beta * x + gamma) *
    (b_poly + beta * k1 * x + gamma) *
    (c_poly + beta * k2 * x + gamma) *
    z_poly)

perm_d = (
    (a_poly + beta * s1_p + gamma) *
    (b_poly + beta * s2_p  + gamma) *
    (c_poly + beta * s3_p + gamma) *
    z_poly(w4*x)
)

t_poly, rem = (gate_part + alpha * (perm_n - perm_d) + alpha^2 * (z_poly - 1)* L1).quo_rem(ZH)
t_lo, t_mid, t_hi = split_t(t_poly, n)
assert t_poly == t_lo + x^(n+2)*t_mid + x^(2*n+4)*t_hi

p_blind = (x^n - 1) * (x - zeta)
t_lo = t_lo + b10 * p_blind
t_mid = t_mid + b11 * p_blind
t_hi = t_hi + b12 * p_blind

t_poly = t_lo + x^(n+2) * t_mid + x^(2*n+4) * t_hi

C_t_lo = commit(t_lo, srs1)
C_t_mid = commit(t_mid, srs1)
C_t_hi = commit(t_hi, srs1)

print(C_t_lo, C_t_mid, C_t_hi)

(26 : 45 : 1) (68 : 74 : 1) (65 : 3 : 1)


In [11]:
# Round 4

a_zeta = a_poly(zeta)
b_zeta = b_poly(zeta)
c_zeta = c_poly(zeta)
s1_zeta = s1_p(zeta)
s2_zeta = s2_p(zeta)
s3_zeta = s3_p(zeta)
z_omega_zeta = z_poly(w4 * zeta)   # z is shifted by w4, matching perm_d

t_zeta = t_lo(zeta) + zeta^(n+2)*t_mid(zeta) + zeta^(2*n+4)*t_hi(zeta)

print(a_zeta, b_zeta, c_zeta, s1_zeta, s2_zeta, s3_zeta, z_omega_zeta, t_zeta)

14 7 1 6 12 16 5 5


In [12]:
r_poly = ( 
    a_zeta * b_zeta * qM_p + 
    a_zeta*qL_p +
    b_zeta*qR_p + 
    c_zeta*qO_p + 
    qC_p
) + alpha * (
        (a_zeta + beta * zeta + gamma) *
        (b_zeta + beta * k1* zeta + gamma) *
        (c_zeta + beta * k2* zeta + gamma) *
        z_poly

        - 
        
        (a_zeta + beta * s1_zeta + gamma) *
        (b_zeta + beta * s2_zeta + gamma) *
        (c_zeta + beta * s3_zeta + gamma) *
        z_omega_zeta
    ) + alpha^2 * L1(zeta) * (z_poly - 1) - ZH(zeta) * (t_lo + zeta^(n+2) * t_mid + zeta^(2*n+4) * t_hi)
        
    

opening_poly = (
    v * (a_poly - a_zeta) +
    v^2 * (b_poly - b_zeta) + 
    v^3 * (c_poly - c_zeta) + 
    v^4 * (s1_p - s1_zeta) +
    v^5 * (s2_p - s2_zeta)
)
print("r(x): ", r_poly)
print(r_poly(zeta))
W_zeta_poly, rem1 = (r_poly + opening_poly).quo_rem(x-zeta)
print(W_zeta_poly , rem1)
W_zeta_omega_poly, rem2 = (z_poly - z_omega_zeta).quo_rem(x - w4 * zeta)

print(rem1, rem2)
assert rem1 == 0 and rem2 == 0

W_zeta = commit(W_zeta_poly, srs1)
W_zeta_omega = commit(W_zeta_omega_poly, srs1)
print(W_zeta, W_zeta_omega)

r(x):  9*x^6 + 3*x^5 + 2*x^4 + x^3 + 2*x^2 + 5*x + 3
0
9*x^5 + 5*x^4 + 10*x^3 + 16*x + 11 0
0 0
(65 : 3 : 1) (0 : 1 : 0)


In [13]:
# Plonk Proof 

print("[a]1 = ", C_a)
print("[b]1 = ", C_b)
print("[c]1 = ", C_c)

print("[z]1 = ", C_z)

print("[t_lo]1 = ", C_t_lo)
print("[t_mid]1 = ", C_t_mid)
print("[t_hi]1 = ", C_t_hi)

print("[W_z]1 = ", W_zeta)
print("[W_zw]1 = ", W_zeta_omega)

print("[a] = ", a_zeta)
print("[b] = ", b_zeta)
print("[c] = ", c_zeta)
print("[s1] = ", s1_zeta)
print("[s2] = ", s2_zeta)
print("[s3] = ", s3_zeta)

print("[zw] = ", z_omega_zeta)

[a]1 =  (26 : 45 : 1)
[b]1 =  (26 : 56 : 1)
[c]1 =  (12 : 69 : 1)
[z]1 =  (1 : 2 : 1)
[t_lo]1 =  (26 : 45 : 1)
[t_mid]1 =  (68 : 74 : 1)
[t_hi]1 =  (65 : 3 : 1)
[W_z]1 =  (65 : 3 : 1)
[W_zw]1 =  (0 : 1 : 0)
[a] =  14
[b] =  7
[c] =  1
[s1] =  6
[s2] =  12
[s3] =  16
[zw] =  5


In [14]:
C_qM = commit(qM_p, srs1)
C_qL = commit(qL_p, srs1)
C_qR = commit(qR_p, srs1)
C_qO = commit(qO_p, srs1)
C_qC = commit(qC_p, srs1)
C_s1 = commit(s1_p, srs1)
C_s2 = commit(s2_p, srs1)
C_s3 = commit(s3_p, srs1)

L1_zeta = L1(zeta)
ZH_zeta = ZH(zeta)
PI_zeta = PI_p(zeta)


In [15]:
D = (
    smul(a_zeta * b_zeta, C_qM) +
    smul(a_zeta, C_qL) + 
    smul(b_zeta, C_qR) + 
    smul(c_zeta, C_qO) +
    C_qC
) + smul((
    (a_zeta + beta * zeta + gamma) *
    (b_zeta + beta * k1 * zeta + gamma) *
    (c_zeta + beta * k2 * zeta + gamma
) * alpha + L1_zeta * (alpha^2) + u), C_z) - smul(ZH_zeta, C_t_lo + smul(zeta^(n+2), C_t_mid) + smul(zeta^(2*n+4), C_t_hi))


F = D + smul(v, C_a) + smul(v^2, C_b) + smul(v^3, C_c) + smul(v^4, C_s1) + smul(v^5, C_s2)

r_o = PI_zeta - L1_zeta * alpha^2 - alpha * (a_zeta + beta * s1_zeta + gamma)*(b_zeta + beta * s2_zeta + gamma)*(c_zeta + beta * s3_zeta + gamma)*z_omega_zeta

Es=-r_o + v * a_zeta + v^2 * b_zeta + v^3 * c_zeta + v^4 * s1_zeta + v^5 * s2_zeta + u*z_omega_zeta

E = smul(Es,G1)

A = W_zeta + smul(u, W_zeta_omega)
B = smul(zeta, W_zeta) + smul(u*zeta*w4, W_zeta_omega) + F - E

In [16]:
A_2 = EK(A)
B_2 = EK(B)
print(A_2.weil_pairing(srs2_s, r))
print(B_2.weil_pairing(srs2_1, r))

37*t + 26
37*t + 26
